In [9]:
%pip install pandas scikit-learn
%pip install ujson

67.96s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


74.00s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


In [10]:
tasks = ["2-way"]#, "3-way"] 
tracks = ["unseen_answers"]#, "unseen_questions"]
#cases = ["gold", "ilponly", "trained", "with_ilp", "with_theory", "with_test_ilp", "with_ilp_and_theory", "with_test_ilp_and_theory"]
cases = ["trained", "with_theory", "with_test_ilp", "with_test_ilp_and_theory"] 

track = tracks[0]
resultsfolder = "results"

In [11]:
import os
import pandas as pd
import ujson

from sklearn.metrics import precision_score, recall_score, f1_score, cohen_kappa_score

In [12]:
# Let's define an ordered category for the prediction:
prediction_type = pd.CategoricalDtype(
    categories=['incorrect', 'partially_correct', 'correct'],
    ordered=True
)
prediction_type

CategoricalDtype(categories=['incorrect', 'partially_correct', 'correct'], ordered=True, categories_dtype=str)

In [13]:
# First, we convert the amati predictions to the format needed for the BEA shared analysis notebook. This is done by extracting the question_id, id, and prediction score for each question and saving it in a new json file in the results folder.
# May as well get the given data while we're at it, to avoid having to read the same files multiple times in the BEA shared analysis notebook.
from unittest import case


for task in tasks:
    hyphenless = task.replace("-", "")
    amati_prefices = [f"train_{hyphenless}", f"trial_{hyphenless}"]
    for prefix in amati_prefices:
        with open(os.path.join(".", f"{task}", f"{prefix}_with_prediction.json"), "r") as f:
            amati_data = ujson.load(f)
            gold_results = []
            amati_results = []
            for item in amati_data:
                gold_results.append({
                    "id": item["id"],
                    "question_id": item["question_id"],
                    "score": item["score"],
                    "explanation": "Given."
                })
                if "amati" in item and "prediction" in item["amati"]:
                    amati_results.append({
                        "id": item["id"],
                        "question_id": item["question_id"],
                        "score": item["amati"]["prediction"],
                        "explanation": "Predicted by ILP theory"
                    })
        with open(os.path.join(".", f"{resultsfolder}", f"{task}_ilponly.json"), "w") as f:
                f.write("{\n")
                f.write(f"    \"task\": \"{task}\",\n")
                f.write(f"    \"case\": \"ilponly\",\n")
                f.write(f"    \"track\": \"{track}\",\n")
                f.write(f"    \"results\": ")
                ujson.dump(amati_results, f)
                f.write("}")
        with open(os.path.join(".", f"{resultsfolder}", f"{task}_gold.json"), "w") as f:
                f.write("{\n")
                f.write(f"    \"task\": \"{task}\",\n")
                f.write(f"    \"case\": \"gold\",\n")
                f.write(f"    \"track\": \"{track}\",\n")
                f.write(f"    \"results\": ")
                ujson.dump(gold_results, f)
                f.write("}")

In [40]:
resultsets = {}

resultsets[task] = {}
resultslist = os.listdir(os.path.join(".", f"{resultsfolder}"))
dataframes = []
gold_dataframes = {}
for filename in resultslist:  
    print(f"Reading file: {filename}")      
    with open(os.path.join(".", f"{resultsfolder}", filename), "r") as f:
        current_results = ujson.load(f)
        if len(current_results["results"])==0:
            print(f"Warning: No results found in file {filename}. Skipping.")
            continue
        current_results_df = pd.DataFrame(current_results["results"])
        current_results_df["case"] = current_results["track"] + "_" + current_results["task"] + "_" + current_results["case"] 
        print(current_results["track"] + "_" + current_results["task"] + "_" + current_results["case"] )
        print(current_results_df.head())
        current_results_df["track"] = current_results["track"]
        if current_results["case"] == "gold":
            gold_dataframes[current_results["track"]] = current_results_df
            gold_dataframes[current_results["track"]].rename(columns={'score': 'actual'}, inplace=True)
        else:
            dataframes.append(current_results_df)
if not gold_dataframes:
    print(f"Error: No gold dataframe found for task {task}. Skipping merging.")
else:
    for df in dataframes:
        try:
            tmpcase = df['case'].iloc[0]
            print(type(df['track'][0]))
        
        
           
            merged_df = pd.merge(gold_dataframes[df["track"].iloc[0]][['id','actual']], df[['id','score']], on="id", )
            resultsets[task][df['case'].iloc[0]] = merged_df
        except Exception as e:
            print(f"Error occurred while merging data for task {task} and case {df['case'].iloc[0]}: {e}")
print("wait")
            
    #

Reading file: 2-way_with_test_ilp_and_theory_responses.json
unseen_answers_2-way_with_test_ilp_and_theory
                                     id                           question_id  \
0  f11b0a02-b253-4448-8c6e-2ad6b464402b  49441f3e-1d4b-4b06-9cd3-b6669884c2df   
1  59d1a944-84c6-4de3-9f4f-402edb20af33  49441f3e-1d4b-4b06-9cd3-b6669884c2df   
2  436bbf8a-3cd2-4c38-9e33-32879397ca32  49441f3e-1d4b-4b06-9cd3-b6669884c2df   
3  ca3ed78c-4f6d-47c5-935f-b9103a9574ec  49441f3e-1d4b-4b06-9cd3-b6669884c2df   
4  93753b7d-6e06-4a29-bca8-ab994bb32c26  35d897f0-c1b4-422c-9a33-43e7da1ffe27   

       score                                        explanation  \
0  Incorrect  The student's answer describes the experimenta...   
1    Correct  The student's answer explicitly compares the t...   
2  Incorrect  The student's answer does not sufficiently com...   
3    Correct  The student's answer correctly compares the tw...   
4  Incorrect  The student's answer describes the presence of...   

    

In [58]:

longest = 0
for case in cases:
    if len(case) > longest:
        longest = len(case)
longest_gap = " " * 2 * (longest + 2)
numwidth = 18
print(f"{longest_gap}\tPrecision{" "*(numwidth-len('Precision'))}\tRecall{" "*(numwidth-len('Recall'))}\tF1 Score{" "*(numwidth-len('F1 Score'))}\tCohen's Kappa Score")
for track in tracks:
    for task in tasks:
        if task not in resultsets:
            print(f"No results found for task {task}. Skipping.")
            continue
        for case in resultsets[task].keys():
            df = resultsets[task][case]
            actuals = df['actual'].str.lower().str.replace('partially correct', 'partially_correct').astype(prediction_type)
            predicteds = df['score'].str.lower().str.replace('partially correct', 'partially_correct').astype(prediction_type)
            precision_value = precision_score(actuals, predicteds, average='weighted')
            recall_value = recall_score(actuals, predicteds, average='weighted')
            f1_score_value = f1_score(actuals, predicteds, average='weighted')
            kappa_score_value = cohen_kappa_score(actuals, predicteds, weights='quadratic')
            print(f"{case}{" "*(2*((longest + 2))-len(case))}\t{precision_value}\t{recall_value}\t{f1_score_value}\t{kappa_score_value}")

                                                    	Precision         	Recall            	F1 Score          	Cohen's Kappa Score
unseen_answers_2-way_with_test_ilp_and_theory       	0.7970395139478245	0.7367758186397985	0.7485526287186095	0.45025144273135054
unseen_answers_2-way_trained                        	0.8546521710131395	0.8362068965517241	0.8408857480378444	0.6300645355767782
unseen_answers_2-way_with_test_ilp                  	0.7632171331865523	0.7592829705505761	0.7610645374879349	0.42519574068274346
unseen_answers_2-way_with_theory                    	0.8437703162916796	0.8290282902829028	0.8332774512640169	0.6090440161492876
unseen_answers_2-way_ilponly                        	0.7994928787671716	0.8068880688806888	0.7953884297965581	0.4855944191220032
